In [ ]:
import torch
import numpy as np
import pandas as pd
import numpy
import os
import re
import json
import pickle
import tokenizers
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

from transformers import BertTokenizer, BertModel, BertForMaskedLM, AutoTokenizer, AutoModelForMaskedLM
from scipy.spatial.distance import cosine
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from nltk.corpus import stopwords
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler


# Load emoji dictionary
with open('Emoji_Dict.p', 'rb') as f:
    emoji_dict = pickle.load(f)

print(f'Loaded {len(emoji_dict)} emojis')
print('Sample emojis:', list(emoji_dict.items())[:5])


def process_emoji_text(text):
    """Replace emojis with their sentiment labels"""
    for emoji, sentiment in emoji_dict.items():
        if emoji in text:
            text = text.replace(emoji, f' [EMOJI_{sentiment.upper()}] ')
    return text


model = AutoModelForMaskedLM.from_pretrained(
    "Shushant/nepaliBERT", output_hidden_states=True, return_dict=True, output_attentions=True)


tokenizers = AutoTokenizer.from_pretrained("Shushant/nepaliBERT")


def get_bert_embedding_sentence(input_sentence, md=model, tokenizer=tokenizers):
    # Process emojis in text
    input_sentence = process_emoji_text(input_sentence)
    marked_text = " [CLS] " + input_sentence + " [SEP] "
    tokenized_text = tokenizer.tokenize(marked_text)

    indexed_tokens = tokenizer.convert_tokens_to_ids(tokenized_text)
    segments_ids = [1] * len(indexed_tokens)

    # Convert inputs to Pytorch tensors
    tokens_tensors = torch.tensor([indexed_tokens])
    segments_tensors = torch.tensor([segments_ids])

    with torch.no_grad():
        outputs = md(tokens_tensors, segments_tensors)
        # removing the first hidden state
        # the first state is the input state

        hidden_states = outputs.hidden_states
    token_vecs = hidden_states[-2][0]
    sentence_embedding = torch.mean(token_vecs, dim=0)
    return sentence_embedding.numpy()

0.8943758573388203
[0]
[1]


In [ ]:
# Emoji-aware sentiment prediction examples
test_sentences = [
    "नराम्रो कुरा नगरेकै बेश 😡",
    "मलाई पढ्न मनपर्छ 😊",
    "यो बहुत राम्रो छ 🎉",
    "मलाई पत्यै मन छैन 😤"
]

print("\n=== Emoji-Aware Sentiment Predictions ===")
for sentence in test_sentences:
    embedding = get_bert_embedding_sentence(sentence)
    pred = svc.predict(np.array(embedding).reshape(1, -1))
    sentiment = "Positive" if pred[0] == 1 else "Negative"
    print(f"Text: {sentence}")
    print(f"Prediction: {sentiment}\n")


# Analyze emoji impact on predictions
print("\n=== Emoji Impact Analysis ===")
text_without_emoji = "नेपाल को संस्कृति राम्रो छ"
text_with_positive_emoji = text_without_emoji + " 😊"
text_with_negative_emoji = text_without_emoji + " 😡"

pred1 = svc.predict(np.array(get_bert_embedding_sentence(text_without_emoji)).reshape(1, -1))[0]
pred2 = svc.predict(np.array(get_bert_embedding_sentence(text_with_positive_emoji)).reshape(1, -1))[0]
pred3 = svc.predict(np.array(get_bert_embedding_sentence(text_with_negative_emoji)).reshape(1, -1))[0]

print(f"Text only: {1 if pred1 == 1 else 0}")
print(f"Text + 😊: {1 if pred2 == 1 else 0}")
print(f"Text + 😡: {1 if pred3 == 1 else 0}")